# MarketPulse — SQL Analytics

Builds the analytical layer on top of the Delta tables saved in `02_spark_etl`: monthly revenue trends, cohort retention, conversion funnel, and customer/product rankings.

Uses Spark SQL directly on the saved Delta tables (`marketpulse_transactions`, `marketpulse_customer_features`, `marketpulse_orders_clean`, `marketpulse_sessions`).

## 1. Monthly Revenue Trend


In [0]:
%sql
WITH order_level AS (
  SELECT DISTINCT order_id, order_purchase_timestamp, total_payment_value
  FROM marketpulse_transactions
)
SELECT
  date_trunc('month', order_purchase_timestamp) AS month,
  ROUND(SUM(total_payment_value), 2) AS revenue,
  COUNT(DISTINCT order_id) AS num_orders
FROM order_level
GROUP BY 1
ORDER BY 1

month,revenue,num_orders
2016-09-01T00:00:00.000Z,null,1
2016-10-01T00:00:00.000Z,46566.71,265
2016-12-01T00:00:00.000Z,19.62,1
2017-01-01T00:00:00.000Z,125584.67,749
2017-02-01T00:00:00.000Z,271298.65,1653
2017-03-01T00:00:00.000Z,414232.07,2543
2017-04-01T00:00:00.000Z,390649.57,2297
2017-05-01T00:00:00.000Z,567066.73,3546
2017-06-01T00:00:00.000Z,490061.61,3133
2017-07-01T00:00:00.000Z,566291.94,3870


In [0]:
%sql
SELECT * FROM marketpulse_transactions
WHERE order_purchase_timestamp >= '2016-09-01' AND order_purchase_timestamp < '2016-10-01'

order_id,product_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,is_delivered,order_item_id,seller_id,shipping_limit_date,price,freight_value,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,total_payment_value,max_installments
bfbd0f9bdef84302105ad712db648a6c,5a6b04657a4c5ee34285d1e4619a96b4,86dc2ffce2dfff336de2f386a786e574,delivered,2016-09-15T12:16:38.000Z,2016-09-15T12:16:38.000Z,2016-11-07T17:11:53.000Z,2016-11-09T07:47:38.000Z,2016-10-04T00:00:00.000Z,true,1,ecccfa2bb93b34a3bf033cc5d1dcdc69,2016-09-19 23:11:33,44.99,2.83,beleza_saude,34.0,1036.0,1.0,1000.0,16.0,16.0,16.0,null,null
bfbd0f9bdef84302105ad712db648a6c,5a6b04657a4c5ee34285d1e4619a96b4,86dc2ffce2dfff336de2f386a786e574,delivered,2016-09-15T12:16:38.000Z,2016-09-15T12:16:38.000Z,2016-11-07T17:11:53.000Z,2016-11-09T07:47:38.000Z,2016-10-04T00:00:00.000Z,true,2,ecccfa2bb93b34a3bf033cc5d1dcdc69,2016-09-19 23:11:33,44.99,2.83,beleza_saude,34.0,1036.0,1.0,1000.0,16.0,16.0,16.0,null,null
bfbd0f9bdef84302105ad712db648a6c,5a6b04657a4c5ee34285d1e4619a96b4,86dc2ffce2dfff336de2f386a786e574,delivered,2016-09-15T12:16:38.000Z,2016-09-15T12:16:38.000Z,2016-11-07T17:11:53.000Z,2016-11-09T07:47:38.000Z,2016-10-04T00:00:00.000Z,true,3,ecccfa2bb93b34a3bf033cc5d1dcdc69,2016-09-19 23:11:33,44.99,2.83,beleza_saude,34.0,1036.0,1.0,1000.0,16.0,16.0,16.0,null,null


In [0]:
%sql
SELECT COUNT(DISTINCT order_id) AS orders_missing_payment
FROM marketpulse_transactions
WHERE total_payment_value IS NULL

orders_missing_payment
1


## 2. Cohort Retention (30/60/90-Day)

Groups customers by the month of their first purchase (their "cohort"), then measures what percentage of each cohort made a repeat purchase within 30, 60, and 90 days of that first order. This answers: "Once someone buys, how many come back — and how quickly?"

In [0]:
%sql
WITH order_level AS (
  SELECT DISTINCT
    t.order_id,
    c.customer_unique_id,
    t.order_purchase_timestamp
  FROM marketpulse_transactions t
  JOIN marketpulse_customers c ON t.customer_id = c.customer_id
),

first_purchase AS (
  SELECT
    customer_unique_id,
    MIN(order_purchase_timestamp) AS first_order_date,
    date_trunc('month', MIN(order_purchase_timestamp)) AS cohort_month
  FROM order_level
  GROUP BY customer_unique_id
),

repeat_flags AS (
  SELECT
    fp.customer_unique_id,
    fp.cohort_month,
    fp.first_order_date,
    MAX(CASE WHEN ol.order_purchase_timestamp > fp.first_order_date
             AND datediff(ol.order_purchase_timestamp, fp.first_order_date) <= 30
        THEN 1 ELSE 0 END) AS repeat_30d,
    MAX(CASE WHEN ol.order_purchase_timestamp > fp.first_order_date
             AND datediff(ol.order_purchase_timestamp, fp.first_order_date) <= 60
        THEN 1 ELSE 0 END) AS repeat_60d,
    MAX(CASE WHEN ol.order_purchase_timestamp > fp.first_order_date
             AND datediff(ol.order_purchase_timestamp, fp.first_order_date) <= 90
        THEN 1 ELSE 0 END) AS repeat_90d
  FROM first_purchase fp
  JOIN order_level ol ON fp.customer_unique_id = ol.customer_unique_id
  GROUP BY fp.customer_unique_id, fp.cohort_month, fp.first_order_date
)

SELECT
  cohort_month,
  COUNT(DISTINCT customer_unique_id) AS cohort_size,
  ROUND(100.0 * SUM(repeat_30d) / COUNT(DISTINCT customer_unique_id), 2) AS retention_30d_pct,
  ROUND(100.0 * SUM(repeat_60d) / COUNT(DISTINCT customer_unique_id), 2) AS retention_60d_pct,
  ROUND(100.0 * SUM(repeat_90d) / COUNT(DISTINCT customer_unique_id), 2) AS retention_90d_pct
FROM repeat_flags
GROUP BY cohort_month
ORDER BY cohort_month

cohort_month,cohort_size,retention_30d_pct,retention_60d_pct,retention_90d_pct
2016-09-01T00:00:00.000Z,1,0.00,0.00,0.00
2016-10-01T00:00:00.000Z,262,1.15,1.15,1.15
2016-12-01T00:00:00.000Z,1,100.00,100.00,100.00
2017-01-01T00:00:00.000Z,716,2.79,3.07,3.07
2017-02-01T00:00:00.000Z,1628,1.29,1.41,1.60
2017-03-01T00:00:00.000Z,2501,1.48,1.72,2.20
2017-04-01T00:00:00.000Z,2250,1.24,1.60,1.87
2017-05-01T00:00:00.000Z,3451,1.80,2.14,2.49
2017-06-01T00:00:00.000Z,3035,1.68,2.11,2.54
2017-07-01T00:00:00.000Z,3750,1.68,2.00,2.19


In [0]:
spark.sql("SELECT * FROM marketpulse_customers LIMIT 5").show()

+--------------------+--------------------+------------------------+-------------+--------------+
|         customer_id|  customer_unique_id|customer_zip_code_prefix|customer_city|customer_state|
+--------------------+--------------------+------------------------+-------------+--------------+
|552215a43eb8963b5...|dbc6ce0dfea576d88...|                   56460|  petrolandia|            PE|
|a6472fafe286c66c4...|8dff9b4552dd014ec...|                   88303|       itajai|            SC|
|d95f60d70d9ea9a7f...|004b45ec5c6418746...|                   57035|       maceio|            AL|
|1aac2b5adf8aec2c0...|96fab6e511253d619...|                    8240|    sao paulo|            SP|
|3ce5345931105e4e5...|7fb6db76737f019f0...|                   12710|     cruzeiro|            SP|
+--------------------+--------------------+------------------------+-------------+--------------+



In [0]:
%sql
SELECT
  COUNT(*) AS total_sessions,
  SUM(CASE WHEN added_to_cart THEN 1 ELSE 0 END) AS added_to_cart,
  SUM(CASE WHEN checkout_started THEN 1 ELSE 0 END) AS checkout_started,
  SUM(CASE WHEN purchase THEN 1 ELSE 0 END) AS purchased,

  ROUND(100.0 * SUM(CASE WHEN added_to_cart THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_session_to_cart,
  ROUND(100.0 * SUM(CASE WHEN checkout_started THEN 1 ELSE 0 END)
        / NULLIF(SUM(CASE WHEN added_to_cart THEN 1 ELSE 0 END), 0), 2) AS pct_cart_to_checkout,
  ROUND(100.0 * SUM(CASE WHEN purchase THEN 1 ELSE 0 END)
        / NULLIF(SUM(CASE WHEN checkout_started THEN 1 ELSE 0 END), 0), 2) AS pct_checkout_to_purchase,
  ROUND(100.0 * SUM(CASE WHEN purchase THEN 1 ELSE 0 END) / COUNT(*), 2) AS overall_conversion_pct
FROM marketpulse_sessions

total_sessions,added_to_cart,checkout_started,purchased,pct_session_to_cart,pct_cart_to_checkout,pct_checkout_to_purchase,overall_conversion_pct
159105,126265,115541,99441,79.36,91.51,86.07,62.50


## 4. Customer & Product Rankings (Window Functions)

Identifies top customers by revenue contribution and top products by category, using window functions (`RANK()`/`ROW_NUMBER()`) rather than simple `ORDER BY ... LIMIT`. This answers two business questions: "who are our most valuable customers?" and "which products drive the most revenue within each category?" — the second one specifically needs a *per-category* ranking, which a single global `LIMIT` can't do.

In [0]:
%sql
WITH order_level AS (
  SELECT DISTINCT order_id, customer_id, total_payment_value
  FROM marketpulse_transactions
)

SELECT
  c.customer_unique_id,
  ROUND(SUM(ol.total_payment_value), 2) AS total_spend,
  COUNT(DISTINCT ol.order_id) AS num_orders
FROM order_level ol
JOIN marketpulse_customers c ON ol.customer_id = c.customer_id
GROUP BY c.customer_unique_id
ORDER BY total_spend DESC
LIMIT 10

customer_unique_id,total_spend,num_orders
0a0a92112bd4c708ca5fde585afaa872,13664.08,1
da122df9eeddfedc1dc1f5349a1a690c,7571.63,2
763c8b1c9c68a0229c42c9fc6f662b93,7274.88,1
dc4802a71eae9be1dd28f5d788ceb526,6929.31,1
459bef486812aa25204be022145caa62,6922.21,1
ff4159b92c40ebe40454e3e6a7c35ed6,6726.66,1
4007669dec559734d6f53e029e360987,6081.54,1
eebb5dda148d3893cdaf5b5ca3040ccb,4764.34,1
48e1ac109decbb87765a3eade6854098,4681.78,1
c8460e4251689ba205045f3ea17884a1,4655.91,4


In [0]:
%sql
WITH product_revenue AS (
  SELECT
    product_category_name,
    product_id,
    SUM(price) AS product_revenue
  FROM marketpulse_transactions
  GROUP BY product_category_name, product_id
),

ranked_products AS (
  SELECT
    *,
    RANK() OVER (PARTITION BY product_category_name ORDER BY product_revenue DESC) AS rank_in_category
  FROM product_revenue
)

SELECT *
FROM ranked_products
WHERE rank_in_category <= 3
ORDER BY product_category_name, rank_in_category

product_category_name,product_id,product_revenue,rank_in_category
agro_industria_e_comercio,11250b0d4b709fee92441c5f34122aed,9111.0,1
agro_industria_e_comercio,423a6644f0aa529e8828ff1f91003690,8043.0,2
agro_industria_e_comercio,672e757f331900b9deea127a2a7b79fd,6885.0,3
alimentos,73326828aa5efe1ba096223de496f596,4379.0199999999995,1
alimentos,89321f94e35fc6d7903d36f74e351d40,3375.709999999999,2
alimentos,ed2067a9c1f79553088a3c67b99a9f97,3187.500000000001,3
alimentos_bebidas,992197904e1d4f0bf3994652373188e4,2586.72,1
alimentos_bebidas,90f97298579cd20412fdcc9b7a2d4b6b,1348.0,2
alimentos_bebidas,84f5c4f480ad6c9998d6a6860f1a2e41,947.16,3
artes,4fe644d766c7566dbc46fb851363cb3b,10603.830000000002,1
